This notebook implements a bronze layer ingestion pipeline for stock market data from Alpha Vantage, following the medallion architecture pattern.
### 
### Architecture:

- Data Source: Alpha Vantage API fetches daily OHLCV (Open, High, Low, Close, Volume) data for AAPL, MSFT, and IBM
- Landing Zone: Raw JSON files land in Volumes/workspace/bronze/landing_zone/stocks/ — one file per ticker per date
- Bronze Table: Auto Loader streams the JSON files into a Lakeflow Spark Declarative Pipeline (DLT) bronze table with deduplication

### Key Features:

- Deduplication: The streaming table architecture prevents duplicates even when the API returns overlapping data (last 100 days on each run)
- Incremental Loading: Auto Loader automatically detects new files and processes only what's new
- Partitioning: Data is partitioned by ticker for efficient querying
- Rate Limiting: 15-second delays between API calls to respect Alpha Vantage limits


- This is a production-ready pattern that handles incremental loads cleanly without manual checkpoint management or duplicate data concerns.


### Reason to use a Streaming Table.

The problem with saving into delta table with .mode("append") on a schedule:\
Every daily run re-fetches the last 100 days and appends all of them →\
 duplicates pile up fast.

### Streaming Table solves this cleanly:

Built-in CDC / APPLY CHANGES INTO deduplicates on a key
Only new rows land, no duplicates. 
- Note:  DLT handles the checkpoint internally. Nothing to declare.

Alpha Vantage API\
      ↓\
  fetch_daily() writes JSON files → /Volumes/.../landing_zone/\
      ↓\
  Auto Loader picks up new files → Streaming Table (Bronze)
  deduplicates on (ticker, date)

In [0]:
# Retrieve the secret code for stocks from Databricks secrets
secret_code = dbutils.secrets.get(scope="stocks", key="api-key")

In [0]:
import time, requests, json
import pandas as pd
from datetime import datetime

api_key = dbutils.secrets.get(scope="stocks", key="api-key")
LANDING_DIR = "/Volumes/workspace/bronze/landing_zone/stocks/"

def fetch_and_land(ticker):
    # Fetch OHLCV and write each day as a JSON file to the landing zone
    url = "https://www.alphavantage.co/query"
    params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": ticker,
        "outputsize": "compact",
        "apikey": api_key,
    }
    data = requests.get(url, params=params).json()

    if "Time Series (Daily)" not in data:
        print(f"[{ticker}] unexpected response: {data}")
        return

    ts = data["Time Series (Daily)"]
    for date, values in ts.items():
        record = {
        "ticker": ticker,
        "date": date,
        "open":         values["1. open"],
        "high":         values["2. high"],
        "low":          values["3. low"],
        "close":        values["4. close"],
        "volume":       values["5. volume"],
        "ingested_at":  datetime.utcnow().isoformat(),
        }
        ts_now = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        path = f"{LANDING_DIR}{ticker}_{date}_{ts_now}.json"
        with open(path, "w") as f:
            json.dump(record, f)

    print(f"[{ticker}] landed {len(ts)} files")

tickers = ["AAPL", "MSFT", "IBM"]
for t in tickers:
    fetch_and_land(t)
    time.sleep(15)


In [0]:
import dlt

@dlt.table(
    comment="Raw stock prices from Alpha Vantage via Auto Loader",
    partition_cols=["ticker"],
    table_properties={
        #"delta.autoOptimize.optimizeOnWrite": "true",
        #"delta.autoOptimize.autoCompact": "true",
    }
)
def stocks_bronze():
    return (
        spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "json")
            #.withColumn("ingested_at", spark.sql.functions.current_timestamp())
            .option("cloudFiles.inferColumnTypes", "true")
            .load("/Volumes/workspace/bronze/landing_zone/stocks/")
    )